# 04 — European Swaption Pricing

**Goal:** take the calibrated Hull–White model and price a simple European payer swaption. We inspect ATM, ITM and OTM strikes to connect model calibration with actual derivative valuation.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import QuantLib as ql

from src.rates_project import *
set_evaluation_date()
print("QuantLib version:", ql.__version__)
print("Evaluation date:", ql.Settings.instance().evaluationDate)

QuantLib version: 1.43
Evaluation date: January 15th, 2026


In [2]:
curve, curve_handle, _ = build_curve()
helpers, swaption_meta = build_swaption_helpers(curve_handle)
model = calibrate_hull_white(curve_handle, helpers)

## 1. Build a 2Y x 5Y ATM payer swaption

In [3]:
atm_swaption, atm_swap, atm_rate, used_strike = make_european_swaption(
    curve_handle, option_years=2, swap_years=5, notional=1_000_000
)
atm_npv = price_swaption_hw(atm_swaption, model)
print(f"ATM forward swap rate: {atm_rate:.6%}")
print(f"Strike:                {used_strike:.6%}")
print(f"Hull–White NPV:        {atm_npv:,.2f}")

ATM forward swap rate: 2.616034%
Strike:                2.616034%
Hull–White NPV:        14,435.12


## 2. Strike sensitivity: ITM / ATM / OTM

For a payer swaption, a lower fixed strike is more valuable than a higher fixed strike, all else equal.

In [4]:
rows = []
for label, strike in [
    ("ITM", atm_rate - 0.0050),
    ("ATM", atm_rate),
    ("OTM", atm_rate + 0.0050),
]:
    swpt, _, _, _ = make_european_swaption(
        curve_handle, option_years=2, swap_years=5,
        notional=1_000_000, strike=strike
    )
    rows.append({"moneyness": label, "strike": strike, "HW_NPV": price_swaption_hw(swpt, model)})
pricing_table = pd.DataFrame(rows)
pricing_table

,moneyness,strike,HW_NPV
0,ITM,0.02116,28102.047580
1,ATM,0.02616,14435.123748
2,OTM,0.03116,6013.434592
